<div style="font-family: 'Times New Roman', Times, serif;">

# Programming Assignment 1
## Distributed Graph Representation and Detecting Network Communities
Due: September 24 2026 by 11:59PM
Submission: via Canvas, individual submission

## 1. Learning Objectives
By completing this assignment, you should be able to:
* Represent and analyze a large, sparse graph with Spark RDDs.
* Translate graph formulas into distributed joins and aggregations.
* Implement local community detection in Spark.

The concepts and datasets are from Yang and Leskovec's SIGKDD paper [1].

## 2. Background
Graph data structure is suitable to represent various natural phenomena in technological, scientific, social, and information systems. Often, vertices in such graph organize into densely linked groups that are commonly referred to as network communities because of the distinctive interactions and associations among the vertices.

### Key concept number 1. Edges and Vertices in the DBLP network
In mathematical notation, a graph is written as:
$$G=(V,E)$$
where V is the set of vertices and E is the set of edges.

Vertices refer to the objects or entities in the graph and edges are the connections or relationships between pairs of vertices.

For the DBLP collaboration network, each vertex represents an author, and an edge connects two authors if they have coauthored at least one paper.

The included DBLP network dataset is an undirected graph. In other words, regardless of each author's contribution, all coauthors are represented based on their coauthorship, and each collaboration on a paper is recorded as an edge without direction.

### Key concept number 2. Weakly Connected Components
A weakly connected component is a maximal group of vertices in a directed graph that are connected when the directions of all edges are ignored. In the DBLP network dataset, you should simply ignore the from and to and consider them as undirected edges. In an undirected graph, such as the DBLP coauthorship network, a weakly connected component is simply called a connected component. Every pair of authors in that component is connected through some sequence of coauthorship relationships.

A component is maximal, meaning that no additional vertex can be added while preserving connectivity.

### Key concept number 3. Average clustering coefficient
The average clustering coefficient measures how strongly the neighbors of vertices in a graph tend to be connected to one another.

For a vertex v with degree $v_k$, its local clustering coefficient is:
$$C_{v}=\frac{2e_{v}}{v_{k}(v_{k}-1)}$$
Where $e_v$ is the number of edges between the neighbors of v.

Equivalently,
$$C_v = \frac{\text{number of connections among v's neighbors}}{\text{maximum possible number of those connections}}$$

The average clustering coefficient of graph $G=(V,E)$ is:
$$C_{avg}=\frac{1}{|V|}\sum_{v\in V}C_{v}$$

Vertices with fewer than two neighbors are typically assigned a clustering coefficient of zero.

For example, suppose author A has three coauthors. Those three coauthors could have at most three connections among themselves:
* If all three have collaborated with one another, $C_{A}=1$.
* If only one pair has collaborated, $C_{A}=1/3$.
* If none has collaborated, $C_{A}=0$.

Therefore, the DBLP network's reported average clustering coefficient of 0.6324 indicates substantial local clustering: an author's coauthors are frequently also coauthors of one another.

### Key Concept number 4. Triangles in a Network
The DBLP network contains 2,224,385 triangles. A triangle consists of three authors who are pairwise connected. In the DBLP coauthorship network, authors A, B, and C form a triangle when all three coauthorship edges exist:
(A, B), (B, C), (A, C)

The number of triangles can be calculated using:
$$T=\frac{1}{3}\sum_{v\in V}t_{v}$$
Where $t_v$ is the number of connected pairs among the neighbors of vertex v. Division by three is necessary because each triangle is counted once at each of its three vertices.

Triangle counts are useful for:
* Measuring clustering: More triangles generally indicate that neighbors frequently connect with one another.
* Identifying communities: Dense communities often contain many triangles.
* Evaluating collaboration patterns: In DBLP, triangles may reveal cohesive groups of researchers.
* Detecting important vertices: Authors participating in many triangles may be central members of research groups.
* Comparing networks: Triangle frequency can distinguish tightly clustered networks from sparse or randomly structured networks.
* Detecting anomalies: An unexpectedly high or low number of triangles may indicate unusual connectivity patterns.

However, the raw triangle count is strongly affected by network size and density. Larger or denser networks naturally tend to contain more triangles. Therefore, it is often more informative to use a normalized measure such as the clustering coefficient or transitivity:
$$\text{Transitivity} = \frac{3 \times \text{number of triangles}}{\text{number of connected triples}}$$

### Key Concept number 5. The fraction of closed triangles
The fraction of closed triangles measures how often an open three vertex pattern is completed to form a triangle. It is also commonly called transitivity or the global clustering coefficient.

A connected triple consists of three vertices connected by at least two edges. If there are two edges, (A, B) and (B, C), it becomes a closed triple when the third edge is present: (A, C).

The fraction is calculated as:
$$\text{Fraction of closed triangles}=\frac{3T}{\sum_{v\in V}\binom{d(v)}{2}}$$
Where T is the number of triangles, $d(v)$ is the degree of vertex v, and $\sum_{v\in V}\binom{d(v)}{2}$ is the number of connected triples.

The factor of 3 appears because each triangle closes one triple centered at each of its three vertices.

For the DBLP network, the reported fraction is 0.1283, meaning that approximately 12.83 percent of connected triples are closed. In practical terms, when an author has collaborated with two other authors, those two coauthors have also collaborated with each other in about 12.83 percent of such configurations.

## 3. Dataset
The DBLP provides a comprehensive list of research papers in computer science. It started in 1993 at the University of Trier in Germany, and it is now operated by the Schloss Dagstuhl Leibniz Center for Informatics. It indexes over 8 million publications from millions of authors, tracking major peer reviewed journals and international conference proceedings.

This assignment includes the DBLP scientific collaboration network where nodes represent authors and edges connect authors that have coauthored a paper. Communities in a scientific domain correspond to people working in common areas and subareas of science.

We will use a curated version of the DBLP network available at the Stanford SNAP graph data dissemination site [2]. Note that the referenced page includes three files. The relevant file is available on the Canvas assignment page. Please use the data file provided there.

## 4. Execution environment and constraints
You will first need to install and configure a Spark cluster on the assigned Linux cluster in CSB. Your first two lab sessions will cover the installation process and the execution of a simple example. For this assignment, please use the same cluster configuration discussed during the lab sessions.

This programming assignment must be implemented exclusively using Apache Spark RDDs. Other APIs, such as DataFrames and Spark SQL, are not permitted. Third party APIs and Spark libraries such as GraphX are also prohibited. If your submission does not meet these requirements, you will receive a score of zero.

There is no specific programming language requirement for this assignment, except that R is not permitted. You may use PySpark, Java, or Scala.

This programming assignment emphasizes your proficiency in a cluster computing environment. Therefore, your software must run correctly on your Apache Spark cluster. If it works correctly only on a single node, a significant number of points will be deducted. For the detailed grading rubric, please refer to Section 7.

## 5. Required tasks and deliverables
All listed tasks must satisfy the constrains specified in Section 4.
</div>

<div style="font-family: 'Times New Roman', Times, serif;">


### Task 1. Graph and community infrastructure
Implement a Spark application to:
* Load the undirected edge list from the provided file into an RDD.
* Identify and remove any self loops.
* Identify and remove any duplicate edges or vertices.
</div>

In [ ]:
from pyspark.sql import SparkSession

def main(sc):
    rdd = sc.textFile("cs435/Lab3_PA1/PA1/com-dblp.ungraph.txt")


if __name__ == "__main__":
    print("I am ready")
    
    spark = SparkSession.builder \
        .appName("PA1_Task1") \
        .getOrCreate()
    sc = spark.sparkContext
    
    main(sc)

note to self: Do not execute this cell, i saved it as html so it would look more convenient
terminal record:
phoenix:~$ start-master.sh
phoenix:~$ start-workers.sh
phoenix:~$ spark-submit cs435/Lab3_PA1/PA1/Ferrell_PA1.py
WARNING: Using incubator modules: jdk.incubator.vector
I am ready
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/21 11:31:04 INFO SparkContext: Running Spark version 4.1.1
26/09/21 11:31:04 INFO SparkContext: OS info Linux, 5.14.0-687.39.1.el9_8.x86_64, amd64
26/09/21 11:31:04 INFO SparkContext: Java version 17.0.20+8-LTS
26/09/21 11:31:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/21 11:31:04 INFO ResourceUtils: ==============================================================
26/09/21 11:31:04 INFO ResourceUtils: No custom resources configured for spark.driver.
26/09/21 11:31:04 INFO ResourceUtils: ==============================================================
26/09/21 11:31:04 INFO SparkContext: Submitted application: PA1_Task1
26/09/21 11:31:04 INFO SecurityManager: Changing view acls to: C837364858
26/09/21 11:31:04 INFO SecurityManager: Changing modify acls to: C837364858
26/09/21 11:31:04 INFO SecurityManager: Changing view acls groups to: C837364858
11:31:06 INFO BlockManagerMasterEndpoint: Registering block manager 129.82.44.160:41073 with 434.4 MiB RAM, BlockManagerId(1, 129.82.44.160, 41073, None)
-skipped for simplicity-
26/09/21 11:31:06 INFO SparkContext: Successfully stopped SparkContext (Uptime: 2141 ms)
phoenix:~$ 

phoenix:~$ head cs435/Lab3_PA1/PA1/com-dblp.ungraph.txt
# Undirected graph: ../../data/output/dblp.ungraph.txt
# DBLP
# Nodes: 317080 Edges: 1049866
# FromNodeId    ToNodeId
0       1
0       2
0       4519
0       23073
0       33043
0       33971

phoenix:~$ tail cs435/Lab3_PA1/PA1/com-dblp.ungraph.txt
425550  425568
425593  425594
425593  425600
425594  425600
425624  425625
425662  425821
425669  425696
425670  425677
425788  425833
425875  425876
phoenix:~$ 

Next Batch of code :


In [ ]:
from pyspark.sql import SparkSession

def main(sc):
    
    # load the text file 
    rdd = sc.textFile("file:///s/chopin/n/under/C837364858/cs435/Lab3_PA1/PA1/com-dblp.ungraph.txt")
    
    
    # pre-processing, filtering out header and comment lines with a hash
    clean_lines = rdd.filter(lambda x: not x.startswith('#'))
    
      
    ''' transform strings into int pairs soruce and destination
    x.split('\t') - takes text str, cuts wherever there is a tab char, turns "0\t1" into  list of two string fragments ['0','1']
    
    map(int,..) task list of strings and runs int function on each fragment , converts text reps into [0,1]
    
    tuple(...) wraps numbers into immutable py tuple => (0,1) 
    
    clean.map(...) spark method takes that transformation logic and applies it across every row of dataset in parallel across cluster executors
    
    '''
    edges = clean_lines.map(lambda x: tuple(map(int, x.split('\t'))))
    
    ''' remove self loops where source equals destination
         x[0] = source author & x[1] = destination author
         lambda fx evals every edge tuple and keeps it only if soure Id is not equal to destination ID. if record exists where author is connecting to selves, x[0]  == x[1], condition = false and spark drops that record 
    '''
    no_selfies = edges.filter(lambda x: x[0] != x[1])
    
    
    # testing
    print("===================================")
    print("first 5 lines of cleaned data: \n")
    print("***************************************************** \n OUTPUT \n" , \
          no_selfies.take(5), \
        "\n ***************************************************** \n ")

if __name__ == "__main__":
    print("I am ready")
    
    spark = SparkSession.builder \
        .appName("PA1_Task1") \
        .getOrCreate()
    sc = spark.sparkContext
    
    main(sc)

phoenix:~$ spark-submit cs435/Lab3_PA1/PA1/Ferrell_PA1.py

26/09/21 11:51:05 INFO ResourceUtils: ==============================================================
26/09/21 11:51:05 INFO ResourceUtils: No custom resources configured for spark.driver.
26/09/21 11:51:05 INFO ResourceUtils: ==============================================================
26/09/21 11:51:05 INFO SparkContext: Submitted application: PA1_Task1
-skipped for simplicity-
26/09/21 11
===================================
first 5 lines of cleaned data: 

***************************************************** 
 OUTPUT 
 [(0, 1), (0, 2), (0, 4519), (0, 23073), (0, 33043)] 
 ***************************************************** 
 

26/09/21 11:51:08 INFO SparkContext: Successfully stopped SparkContext (Uptime: 3663 ms)
phoenix:~$ 

Output was : [(0, 1), (0, 2), (0, 4519), (0, 23073), (0, 33043)]


text file is successfully parsed into clean, proper integer tuples across the cluster.

next piece of task one is making sure edges are fully bidirectional (undirected) and that all duplicate listings are stripped out. in lab 3, if author 0 is connected to author 1, need both (0,1) and (1,0) so labels can flow back and forth during label propagation later. 


In [ ]:
from pyspark.sql import SparkSession

def main(sc):
    
    # load the text file 
    rdd = sc.textFile("file:///s/chopin/n/under/C837364858/cs435/Lab3_PA1/PA1/com-dblp.ungraph.txt")
    
    # pre-processing, filtering out header and comment lines with a hash
    clean_lines = rdd.filter(lambda x: not x.startswith('#'))
    
    # transform strings into int pairs soruce and destination
    edges = clean_lines.map(lambda x: tuple(map(int, x.split('\t'))))
    
    # remove self loops where source equals destination
    no_selfies = edges.filter(lambda x: x[0] != x[1])
    
    '''ensure bidrectionality
        flatmap vs map
        std map op takes one input and returns one output 1:1
        
        flatmap 1:2 ratio, every edge tuple coming in -> output a list containing two tuples
        flatmap auto flattens lists so RDD stays clean collection of edge tuples 
        
    x reps individual edge tuple such as (0,1)
    
    fx returns list w/ 2 items : 
    x: original tuple (0,1)
    (x[1], x[0]) : reversed edge tuple where source and destination swapped becoming (1,0)
    
    undirected graph, edge might only be listed once in txt file in single direction
    if author[0] collabs with author[1], graph als need to know commutative | symmetrical
    '''
    bi_edges = no_selfies.flatMap(lambda x: [x, (x[1], x[0])])
    
    ''' remove duplicate entires using distinct() and save computation with persist()
    
        so far i expanded every edge to go in both directions, creating pairs that might already exist or overlap. using the distinct method i can collapse those down so eveyr unique connection exists only once in dataset. 
        
        in task 2, i will run iterative loops that repeatedly query the edges. persist will prevent recalculating entire lineage on every loop iteration. persisting tells spark to cache results in memory so iterative steps can pull data directly from RAM instead. 
    '''
    unique_edges = bi_edges.distinct().persist()
    
    
    # testing
    print("===================================")
    print("first 5 lines of cleaned data: \n")
    print("***************************************************** \n OUTPUT \n" , \
          no_selfies.take(5), \
        "\n ***************************************************** \n ")
    
    print("===================================")
    print("first 5 unique bi-edges: \n")
    print("***************************************************** \n OUTPUT \n" , \
          unique_edges.take(5), \
        "\n ***************************************************** \n ")


if __name__ == "__main__":
    print("I am ready")
    
    spark = SparkSession.builder \
        .appName("PA1_Task1") \
        .getOrCreate()
    sc = spark.sparkContext
    
    main(sc)

phoenix:~$ spark-submit cs435/Lab3_PA1/PA1/Ferrell_PA1.py
WARNING: Using incubator modules: jdk.incubator.vector
I am ready

===================================
first 5 lines of cleaned data: 

26/09/21 12:11:01 INFO DAGScheduler: Job 0 finished: runJob at PythonRDD.scala:191, took 1290.688857 ms
***************************************************** 
 OUTPUT 
 [(0, 1), (0, 2), (0, 4519), (0, 23073), (0, 33043)] 
 ***************************************************** 
 
===================================
first 5 unique bi-edges: 

26/09/21 12:11:05 INFO DAGScheduler: Job 1 finished: runJob at PythonRDD.scala:191, took 4380.498558 ms
***************************************************** 
 OUTPUT 
 [(0, 2), (2, 0), (0, 120044), (120044, 0), (0, 123880)] 
 ***************************************************** 
 


In [ ]:
from pyspark.sql import SparkSession

def main(sc):
    
    # load the text file 
    rdd = sc.textFile("file:///s/chopin/n/under/C837364858/cs435/Lab3_PA1/PA1/com-dblp.ungraph.txt")
    
    # pre-processing, filtering out header and comment lines with a hash
    clean_lines = rdd.filter(lambda x: not x.startswith('#'))
    
    # transform strings into int pairs soruce and destination
    # edges = clean_lines.map(lambda x: tuple(map(int, x.split('\t'))))
    
    # remove self loops where source equals destination
    # no_selfies = edges.filter(lambda x: x[0] != x[1])
    
    no_selfies = clean_lines.map(lambda x: tuple(map (int, x.split('\t')))).filter(lambda x: x[0] != x[1])
    
    #ensure bidrectionality
    bi_edges = no_selfies.flatMap(lambda x: [x, (x[1], x[0])])
    
    # remove duplicate entires using distinct() 
    unique_edges = bi_edges.distinct().persist()
    
    
    # testing
    print("===================================\n", "first 5 lines of cleaned data: \n", "***************************************************** \n OUTPUT \n" , no_selfies.take(5),"\n ***************************************************** \n ")
    
    print("===================================\n","first 5 unique bi-edges: \n","***************************************************** \n OUTPUT \n",unique_edges.take(5), "\n ***************************************************** \n ")
    
    total_lines = rdd.count()
    #self_loop_count = edges.filter(lambda x: x[0] == x[1]).count()
    self_loop_count = no_selfies.filter(lambda x: x[0] == x[1]).count()
    edge_count = unique_edges.count()
    print(f"**************************************** \n Task one metrics: \n - Total lines: {total_lines} \n - self-loops removed: {self_loop_count} \n - total unique bidirectional edges: {edge_count} \n *************************************" )
    
if __name__ == "__main__":
    print("I am ready")
    
    spark = SparkSession.builder \
        .appName("PA1_Task1") \
        .getOrCreate()
    sc = spark.sparkContext
    
    main(sc)

**Task 1 is complete. loaded massive file across cluster cleaned out comments deleted self loops expanded edge to be bi-dir. and dropped duplicates.**

<div style="font-family: 'Times New Roman', Times, serif;">


### Task 2. Explore and characterize the graph
Implement Spark applications to calculate and report:
* The number of unique nodes and edges.
* The total number of nodes that belong to the single largest interconnected group in this network graph.
* The total number of edges that belong to the single largest interconnected group in this network graph.
* Average clustering coefficient.
* The count of triangles.
* Fraction of closed triangles.

Your grading interview may include questions about:
* Implementation details.
* The concepts used to complete the tasks above and their implications for the DBLP coauthorship network.
* A demonstration of your applications.
* Challenges encountered and potential improvements to your implementation.

</div>

In [ ]:
from pyspark.sql import SparkSession
import sys


def main(sc):
    
    input_path = "file:///" + sys.argv[1]
    # load the text file 
    rdd = sc.textFile(input_path)
    
    # pre-processing, filtering out header and comment lines with a hash
    clean_lines = rdd.filter(lambda x: not x.startswith('#'))
    
    # transform strings into int pairs soruce and destination
    # edges = clean_lines.map(lambda x: tuple(map(int, x.split('\t'))))
    
    # remove self loops where source equals destination
    # no_selfies = edges.filter(lambda x: x[0] != x[1])
    
    no_selfies = clean_lines.map(lambda x: tuple(map (int, x.split('\t')))).filter(lambda x: x[0] != x[1])
    
    #ensure bidrectionality
    bi_edges = no_selfies.flatMap(lambda x: [x, (x[1], x[0])])
    
    # remove duplicate entires using distinct() 
    unique_edges = bi_edges.distinct().persist()
    
    # grouping => edge_group : (author, neighbors_iterable)
    edge_group = unique_edges.groupByKey()
    
    # computing node degrees
    degrees = edge_group.map(lambda x: (x[0], len(x[1])))
    
    
    # testing
    print("===================================\n", "first 5 lines of cleaned data: \n", "***************************************************** \n OUTPUT \n" , no_selfies.take(5),"\n ***************************************************** \n ")
    
    print("===================================\n","first 5 unique bi-edges: \n","***************************************************** \n OUTPUT \n",unique_edges.take(5), "\n ***************************************************** \n ")
    
    total_lines = rdd.count()
    #self_loop_count = edges.filter(lambda x: x[0] == x[1]).count()
    self_loop_count = no_selfies.filter(lambda x: x[0] == x[1]).count()
    edge_count = unique_edges.count()
    print(f"**************************************** \n Task one metrics: \n - Total lines: {total_lines} \n - self-loops removed: {self_loop_count} \n - total unique bidirectional edges: {edge_count} \n *************************************" )
    
    print(
        "********************* \n first 5 adjacency lists: \n *************** \n OUTPUT \n", edge_group.take(5), " \n ******************"
    )
    
    print(
        "*********************\n first 5 node degrees: \n", degrees.take(5), "********************"
    )
    
if __name__ == "__main__":
    print("I am ready")
    
    spark = SparkSession.builder \
        .appName("PA1_Task1") \
        .getOrCreate()
    sc = spark.sparkContext
    
    main(sc)

********************* 
 first 5 adjacency lists: 
 *************** 
 OUTPUT 
 [(112424, <pyspark.resultiterable.ResultIterable object at 0x7f1628e9c810>), (112426, <pyspark.resultiterable.ResultIterable object at 0x7f1628e6c850>), (112428, <pyspark.resultiterable.ResultIterable object at 0x7f1628e9c910>), (112430, <pyspark.resultiterable.ResultIterable object at 0x7f1628e9c310>), (123918, <pyspark.resultiterable.ResultIterable object at 0x7f1628e9ca10>)]  
 ******************
26/09/21 21:18:04 INFO SparkContext: Starting job: runJob at PythonRDD.scala:191

groupbykey is lazy - it returns a pointer instead of a list

if i wrap it in a list() it should pull those items out of the stream and pack them into std py list. the first bit is fine, so .map(lambda x : (x[0], list(x[1]))).take(5) should work

In [ ]:
from pyspark.sql import SparkSession
import sys


def main(sc):
    
    input_path = "file:///" + sys.argv[1]
    # load the text file 
    rdd = sc.textFile(input_path)
    
    # pre-processing, filtering out header and comment lines with a hash
    clean_lines = rdd.filter(lambda x: not x.startswith('#'))
    
    # transform strings into int pairs soruce and destination
    # edges = clean_lines.map(lambda x: tuple(map(int, x.split('\t'))))
    
    # remove self loops where source equals destination
    # no_selfies = edges.filter(lambda x: x[0] != x[1])
    
    no_selfies = clean_lines.map(lambda x: tuple(map (int, x.split('\t')))).filter(lambda x: x[0] != x[1])
    
    #ensure bidrectionality
    bi_edges = no_selfies.flatMap(lambda x: [x, (x[1], x[0])])
    
    # remove duplicate entires using distinct() 
    unique_edges = bi_edges.distinct().persist()
    
    # grouping => edge_group : (author, neighbors_iterable)
    edge_group = unique_edges.groupByKey()
    
    # computing node degrees
    degrees = edge_group.map(lambda x: (x[0], len(x[1])))
    
      
    total_lines = rdd.count()
    #self_loop_count = edges.filter(lambda x: x[0] == x[1]).count()
    self_loop_count = no_selfies.filter(lambda x: x[0] == x[1]).count()
    edge_count = unique_edges.count()  
    
    # testing
    print(
        f"\n===================================\n",
        "first 5 lines of cleaned data: \n\n",
        no_selfies.take(5),
        "\n===================================\n",
        "first 5 unique bi-edges: \n\n",
        unique_edges.take(5),
        "\n=================================== \n Task one metrics: \n\n - Total lines: {total_lines} \n - self-loops removed: {self_loop_count} \n - total unique bidirectional edges: {edge_count} \n ", 
        "\n===================================\n first 5 adjacency lists: \n \n", edge_group.map(lambda x: (x[0], list(x[1]))).take(5), " \n ",
        "\n===================================\n first 5 node degrees: \n\n", 
        degrees.take(5), 
        "\n===================================\n"
    )
    
if __name__ == "__main__":
    print("I am ready")
    
    spark = SparkSession.builder \
        .appName("PA1_Task1") \
        .getOrCreate()
    sc = spark.sparkContext
    
    main(sc)

In [ ]:
===================================
 first 5 lines of cleaned data: 

 [(0, 1), (0, 2), (0, 4519), (0, 23073), (0, 33043)] 
===================================
 first 5 unique bi-edges: 

 [(0, 2), (2, 0), (0, 120044), (120044, 0), (0, 123880)] 
=================================== 
 Task one metrics: 

 - Total lines: {total_lines} 
 - self-loops removed: {self_loop_count} 
 - total unique bidirectional edges: {edge_count} 
  
===================================
 first 5 adjacency lists: 
 
 [(0, [2, 120044, 123880, 124002, 274042, 369692, 413808, 1, 4519, 23073, 33043, 33971, 75503, 101215, 206567, 411025]), (2, [0, 6786, 33126, 37678, 44724, 75502, 79494, 93250, 97758, 108350, 108790, 139006, 157374, 189316, 1, 10077, 16803, 23595, 33043, 54787, 65297, 66305, 75213, 75503, 84203, 127771]), (120044, [0, 93422, 100010, 100800, 112386, 124002, 161140, 196242, 240272, 118303, 120045, 123871, 126567, 147255, 206567, 240271, 250519, 306807, 379551, 389113, 19689, 23617, 37693, 68705, 108089]), (123880, [0, 125170, 274042, 369692, 254939, 287903, 4519, 35367, 75503]), (124002, [0, 120044, 121614, 161584, 360948, 386544, 118303, 121615, 147255, 206567, 243469, 244737, 271951])]  
  
===================================
 first 5 node degrees: 

 [(0, 16), (2, 26), (120044, 25), (123880, 9), (124002, 13)] 
===================================

yay :D

# Create helper functions and restructuring

### label propagating should be a separate helper function and I should consider returning values with main so all print statements are executing separately.

`labels = edge_group.map(lambda x: (x[0], x[0]))`
This creates starting state every key is a node ID and its initial value is its own ID.

`for _ in range(max_iterations):`
instead of checking for convergence after every round, run a fixed number of steps.

15 iterations
Graph algorithms depend on the network diameter which is the longest shortest path between any two nodes. for DBLP, info can travel across the graph within a predictable number of hops. 15 to 20 iterations are enough for the lowest component IDs to reach every single node in the network

`joined = edge_group.join(labels)`
edge_group RDD holds (node_id, [neighbor_list]), and labels RDD holds (node_id, current_label), joining them matches each node up with its neighbor list and its active label all in one tuple: `(node_id, ([neighbor_list], current_label))`

`messages = joined.flatMap(...)`
for every node it unpacks tuple:

    x[0] is the node ID

    x[1][0] is the neighbor list

    x[1][1] is the current label

 emits a pair back to node (node_id, current_label) it remembers its own label and a pair for every single neighbor (neighbor_id, current_label) `flatMap` flattens all messages across cluster into one RDD of (receiver_id, label) pairs

`labels = messages.reduceByKey(min)`
every node receives a pile of incoming labels from itself and its neighbors. reduceByKey(min) shuffles those messages by the receiver ID and keeps only the smallest numerical label out of the entire bunch becoming new labels RDD for the next loop


**Addedd TODO's so I can cont where i left off tomorrow**



In [ ]:
from pyspark.sql import SparkSession
import sys


    

def main(sc):
    
    input_path = "file:///" + sys.argv[1]
    # load the text file 
    rdd = sc.textFile(input_path)
    
    # pre-processing, filtering out header and comment lines with a hash
    no_selfies = rdd.filter(lambda x: not x.startswith('#')) \
        .map(lambda x: tuple(map (int, x.split('\t')))) \
        .filter(lambda x: x[0] != x[1])
    
    # transform strings into int pairs soruce and destination
    # edges = clean_lines.map(lambda x: tuple(map(int, x.split('\t'))))
    
    # remove self loops where source equals destination
    # no_selfies = edges.filter(lambda x: x[0] != x[1])
    
    #ensure bidrectionality
    bi_edges = no_selfies.flatMap(lambda x: [x, (x[1], x[0])])
    
    # remove duplicate entires using distinct() 
    unique_edges = bi_edges.distinct().persist()
    
    # grouping => edge_group : (author, neighbors_iterable)
    edge_group = unique_edges.groupByKey()
    
    # computing node degrees
    degrees = edge_group.map(lambda x: (x[0], len(x[1])))
    

      
    total_lines = rdd.count()
    #self_loop_count = edges.filter(lambda x: x[0] == x[1]).count()
    self_loop_count = no_selfies.filter(lambda x: x[0] == x[1]).count()
    edge_count = unique_edges.count()  
    
    # run label prop to get component labels: (node_id, component_id)
    component_labels = label_prop(edge_group)
    
    #TODO: find size of largest component, flip to (component_id, 1) and sum
    #TODO: find max component size
    
    return {
        "no_selfies": no_selfies,
        "unique_edges": unique_edges,
        "edge_group": edge_group,
        "degrees": degrees,
        "total_lines": total_lines,
        "self_loop_count": self_loop_count,
        "edge_count": edge_count,
       # "largest_component_size": largest_component_size
    }
    
'''
label_propagation is an algorithm that takes in the adjacency list and returns
edge_group :
'''
def label_prop(edge_group, max_iterations=15):
    
    labels = edge_group.map(lambda x: (x[0], x[0]))
    
    for _ in range(max_iterations):
        joined = edge_group.join(labels)
        messages = joined.flatMap(
            lambda x: [(x[0], x[1][1])]
            + [(neighbor, x[1][1]) for neighbor in x[1][0]]
    )
        labels = messages.reduceByKey(min)
    return labels




if __name__ == "__main__":
    print("I am ready")
    
    spark = SparkSession.builder \
        .appName("PA1_Task1") \
        .getOrCreate()
    sc = spark.sparkContext
    metrics = main(sc)
    print(
        f"\n===================================\n",
        "first 5 lines of cleaned data: \n\n",
        metrics["no_selfies"].take(5),
        "\n===================================\n",
        "first 5 unique bi-edges: \n\n",
        metrics["unique_edges"].take(5),
        "\n=================================== \n Task one metrics: \n\n",
        f" - Total lines: {metrics['total_lines']} \n",
        f" - self-loops removed: {metrics['self_loop_count']} \n",
        f" - total unique bidirectional edges: {metrics['edge_count']} \n",
        "\n===================================\n first 5 adjacency lists: \n \n",
        metrics["edge_group"].map(lambda x: (x[0], list(x[1]))).take(5), " \n ",
        "\n===================================\n first 5 node degrees: \n\n",
        metrics["degrees"].take(5),
        "\n===================================\n"#,
        #f"Largest WCC node count: {metrics['largest_component_size']} \n",
       # "===================================\n"
    )
    
    

<div style="font-family: 'Times New Roman', Times, serif;">


### Bonus Task
Calculate and report the diameter of this network.

## 6. Minimum correctness check
Most of the correct answers are available in the web page of the provided dataset [2]. You can compare your answers with listed statistics for the graph.

## 7. AI usage guideline
You may use AI agents to assist with coding. We will provide guidelines for using them to generate better code while enhancing your learning.

## 8. Grading rubrics
* **Completeness of your code:** 1 percent towards course grade. Your code must be executed correctly on an Apache Spark cluster in CSB with at least 2 Linux nodes. Your output should be correct.
* **Task 1:** 1 percent towards course grade. In the grading interview, you should be able to answer all the questions correctly.
* **Task 2:** 3 percent towards course grade. In the grading interview, you should be able to answer all the questions correctly.
* **Bonus task:** 1 percent towards course grade. In the grading interview, you should be able to answer all the questions correctly.

The late submission policy is available in the course syllabus.

## References
[1] Yang, Jaewon, and Jure Leskovec. Defining and evaluating network communities based on ground truth. In Proceedings of the ACM SIGKDD workshop on mining data semantics, pages 1 to 8, 2012.
[2] Stanford Network Analysis Project. DBLP collaboration network and ground truth communities. Stanford University. Retrieved August 31, 2026, from https://snap.stanford.edu/data/com-DBLP.html

</div>
 